In [20]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Sakriva INFO i WARNING poruke
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense

In [4]:
!ls -lh /storage/train_cifar100_pilot18.py

-rw-r--r-- 1 root root 17K Sep 22 20:23 /storage/train_cifar100_pilot18.py


In [6]:
!grep -n "full216" /storage/train_cifar100_pilot18.py

In [7]:
!grep -n "full216" /storage/train_cifar100_pilot18.py

In [8]:
!ls -lh /storage/train_cifar100_pilot18.py
!grep -n "full216" /storage/train_cifar100_pilot18.py

-rw-r--r-- 1 root root 17K Sep 22 20:23 /storage/train_cifar100_pilot18.py


In [9]:
!find /notebooks /storage -maxdepth 2 -type f \
  -name "train_cifar100_pilot18*.py" \
  -printf "%TY-%Tm-%Td %TH:%TM  %s bytes  %p\n"

2026-09-23 22:21  18866 bytes  /notebooks/train_cifar100_pilot18.py
2026-09-22 20:23  16927 bytes  /storage/train_cifar100_pilot18.py


In [10]:
!cp /notebooks/train_cifar100_pilot18.py \
  /storage/train_cifar100_pilot18.py

In [11]:
!ls -lh /storage/train_cifar100_pilot18.py
!grep -n "full216" /storage/train_cifar100_pilot18.py

-rw-r--r-- 1 root root 19K Sep 23 22:22 /storage/train_cifar100_pilot18.py
67:        return "full216"


In [12]:
!python /storage/train_cifar100_pilot18.py \
  --prepare-only \
  --pilot-size 216 \
  --workers 10

Pilot assignments saved: /storage/cnn_topology_study_same216/cifar100_full216_assignments.csv
Architectures per worker:
worker_id
1     22
2     22
3     22
4     22
5     22
6     22
7     21
8     21
9     21
10    21
Parameter-load proxy per worker:
worker_id
1     2613336
2     2611544
3     2611928
4     2610392
5     2612696
6     2613336
7     2572116
8     2571988
9     2573140
10    2571860
Selected architecture IDs:
 worker_id architecture_id                       model_name  avg_degree  degree_entropy  assortativity_coefficient
         1            A007          std_std_std_k3_5_3_same  679.155303        3.305865                  -0.580873
         1            A011       std_std_group4_k3_5_3_same  422.670455        3.158403                  -0.384155
         1            A021    std_group2_group2_k3_5_3_same  388.387626        3.202957                  -0.532736
         1            A044          std_std_std_k5_5_5_same 1372.488636        3.947036                  -0.28

In [13]:
!python /storage/train_cifar100_pilot18.py \
  --worker-id 1 \
  --workers 10 \
  --pilot-size 216 \
  --max-new-runs 1

Reused 15 completed pilot run(s).
Worker 1: 22 architectures, 110 total seed runs.
Data: train=45000, validation=5000, test=10000
Starting worker=1, architecture=A007, seed=11
I0000 00:00:1790202507.991468     673 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
Finished A007, seed=11: test_acc=0.4730, best_epoch=46, time=2.6 min
Stopped after 1 new run(s) as requested. Run again without --max-new-runs to resume.


In [14]:
!python /storage/train_cifar100_pilot18.py \
  --worker-id 1 \
  --workers 10 \
  --pilot-size 216

Reused 15 completed pilot run(s).
Worker 1: 22 architectures, 110 total seed runs.
Data: train=45000, validation=5000, test=10000
Starting worker=1, architecture=A007, seed=22
I0000 00:00:1790202929.526830    1511 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
Finished A007, seed=22: test_acc=0.4827, best_epoch=52, time=2.9 min
Starting worker=1, architecture=A007, seed=33
Finished A007, seed=33: test_acc=0.4857, best_epoch=42, time=2.4 min
Starting worker=1, architecture=A007, seed=44
Finished A007, seed=44: test_acc=0.4848, best_epoch=47, time=2.6 min
Starting worker=1, architecture=A007, seed=55
Finished A007, seed=55: test_acc=0.4741, best_epoch=58, time=3.1 min
Starting worker=1, architecture=A021, seed=11
Finished A021, seed=11: test_acc=0.4584, best_epoch=74, time=8.5 min
Starting worker=1, architecture=A021, seed=22
Finished A021, seed=22: test_acc=0.4647, best_epoch=55, time=6.6 min
Starting worker=1, archi

In [15]:
from pathlib import Path
import pandas as pd

mapa = Path(
    "/storage/cnn_topology_study_same216/"
    "training_cifar100_full216"
)

datoteke = sorted(mapa.glob("worker_*_runs.csv"))
print("Broj worker datoteka:", len(datoteke))

svi = []

for putanja in datoteke:
    dio = pd.read_csv(putanja)
    duplikati = dio.duplicated(
        ["architecture_id", "seed"]
    ).sum()

    print(
        f"{putanja.name}: "
        f"redova={len(dio)}, "
        f"arhitektura={dio['architecture_id'].nunique()}, "
        f"duplikata={duplikati}"
    )

    svi.append(dio)

rezultati = pd.concat(svi, ignore_index=True)

raspodjela = pd.read_csv(
    "/storage/cnn_topology_study_same216/"
    "cifar100_full216_assignments.csv"
)

seedovi = {11, 22, 33, 44, 55}

ocekivani_parovi = {
    (str(arhitektura), seed)
    for arhitektura in raspodjela["architecture_id"]
    for seed in seedovi
}

stvarni_parovi = set(
    zip(
        rezultati["architecture_id"].astype(str),
        rezultati["seed"].astype(int),
    )
)

broj_seedova = rezultati.groupby(
    "architecture_id"
)["seed"].nunique()

print("\nUkupno redova:", len(rezultati))
print(
    "Broj arhitektura:",
    rezultati["architecture_id"].nunique()
)
print(
    "Ukupno duplikata:",
    rezultati.duplicated(
        ["architecture_id", "seed"]
    ).sum()
)
print("Nedostajućih treninga:", len(ocekivani_parovi - stvarni_parovi))
print("Neočekivanih treninga:", len(stvarni_parovi - ocekivani_parovi))
print("Arhitektura bez 5 seedova:", (broj_seedova != 5).sum())
print(
    "Nedostajućih testnih rezultata:",
    rezultati[["test_acc", "test_loss"]].isna().any(axis=1).sum()
)

assert len(datoteke) == 10
assert len(rezultati) == 1080
assert rezultati["architecture_id"].nunique() == 216
assert rezultati.duplicated(["architecture_id", "seed"]).sum() == 0
assert ocekivani_parovi == stvarni_parovi
assert (broj_seedova == 5).all()
assert not rezultati[["test_acc", "test_loss"]].isna().any().any()

print("\nPROVJERA USPJEŠNA: 216 × 5 = 1080 treninga.")

Broj worker datoteka: 10
worker_01_runs.csv: redova=110, arhitektura=22, duplikata=0
worker_02_runs.csv: redova=110, arhitektura=22, duplikata=0
worker_03_runs.csv: redova=110, arhitektura=22, duplikata=0
worker_04_runs.csv: redova=110, arhitektura=22, duplikata=0
worker_05_runs.csv: redova=110, arhitektura=22, duplikata=0
worker_06_runs.csv: redova=110, arhitektura=22, duplikata=0
worker_07_runs.csv: redova=105, arhitektura=21, duplikata=0
worker_08_runs.csv: redova=105, arhitektura=21, duplikata=0
worker_09_runs.csv: redova=105, arhitektura=21, duplikata=0
worker_10_runs.csv: redova=105, arhitektura=21, duplikata=0

Ukupno redova: 1080
Broj arhitektura: 216
Ukupno duplikata: 0
Nedostajućih treninga: 0
Neočekivanih treninga: 0
Arhitektura bez 5 seedova: 0
Nedostajućih testnih rezultata: 0

PROVJERA USPJEŠNA: 216 × 5 = 1080 treninga.


In [16]:
from pathlib import Path
from itertools import combinations
import json
import numpy as np
import pandas as pd

study = Path("/storage/cnn_topology_study_same216")
seed = 20260917
test_size = 54

arhitekture = pd.read_csv(
    study / "cifar100_full216_assignments.csv"
)

pilot = pd.read_csv(
    study / "cifar100_pilot18_assignments.csv"
)

pilot_ids = set(pilot["architecture_id"].astype(str))
arhitekture["architecture_id"] = (
    arhitekture["architecture_id"].astype(str)
)

pilot_mask = arhitekture["architecture_id"].isin(pilot_ids).to_numpy()
dozvoljeni = np.flatnonzero(~pilot_mask)

faktori = [
    "conv1", "conv2", "conv3",
    "kernel1", "kernel2", "kernel3",
    "worker_id",
]

grupe = []

def dodaj_grupu(vrijednosti, tezina=1.0):
    kodovi, razine = pd.factorize(
        pd.Series(vrijednosti).astype(str),
        sort=True,
    )
    cilj = (
        np.bincount(kodovi, minlength=len(razine))
        * test_size / len(arhitekture)
    )
    grupe.append((kodovi, len(razine), cilj, tezina))

for faktor in faktori:
    dodaj_grupu(arhitekture[faktor], 1.0)

for prvi, drugi in combinations(faktori[:6], 2):
    kombinacija = (
        arhitekture[prvi].astype(str)
        + "|"
        + arhitekture[drugi].astype(str)
    )
    dodaj_grupu(kombinacija, 0.20)

def ocjena(indeksi):
    rezultat = 0.0
    for kodovi, broj_razina, cilj, tezina in grupe:
        dobiveno = np.bincount(
            kodovi[indeksi],
            minlength=broj_razina,
        )
        rezultat += tezina * np.sum(
            (dobiveno - cilj) ** 2 / (cilj + 1e-12)
        )
    return rezultat

rng = np.random.default_rng(seed)
najbolji = None
najbolja_ocjena = np.inf

for _ in range(50000):
    kandidat = rng.choice(
        dozvoljeni,
        size=test_size,
        replace=False,
    )
    vrijednost = ocjena(kandidat)

    if vrijednost < najbolja_ocjena:
        najbolji = kandidat.copy()
        najbolja_ocjena = vrijednost

arhitekture["analysis_split"] = "development"
arhitekture.loc[najbolji, "analysis_split"] = "final_holdout"
arhitekture["used_in_pilot"] = pilot_mask

izlaz = study / "cifar100_full216_analysis_split.csv"
arhitekture.to_csv(izlaz, index=False)

assert (arhitekture["analysis_split"] == "development").sum() == 162
assert (arhitekture["analysis_split"] == "final_holdout").sum() == 54
assert not arhitekture.loc[
    arhitekture["analysis_split"] == "final_holdout",
    "used_in_pilot",
].any()

print(arhitekture["analysis_split"].value_counts())
print("Pilot-arhitektura u završnom skupu: 0")
print("Spremljeno:", izlaz)

for faktor in faktori:
    print("\n", faktor)
    print(pd.crosstab(
        arhitekture[faktor],
        arhitekture["analysis_split"],
    ))

analysis_split
development      162
final_holdout     54
Name: count, dtype: int64
Pilot-arhitektura u završnom skupu: 0
Spremljeno: /storage/cnn_topology_study_same216/cifar100_full216_analysis_split.csv

 conv1
analysis_split  development  final_holdout
conv1                                     
group2                   55             17
group4                   53             19
std                      54             18

 conv2
analysis_split  development  final_holdout
conv2                                     
group2                   53             19
group4                   54             18
std                      55             17

 conv3
analysis_split  development  final_holdout
conv3                                     
group2                   53             19
group4                   53             19
std                      56             16

 kernel1
analysis_split  development  final_holdout
kernel1                                   
3                        81   

In [17]:
from pathlib import Path
import pandas as pd

study = Path("/storage/cnn_topology_study_same216")
run_dir = study / "training_cifar100_full216"
archive = study / "analysis_cifar100_full216"
archive.mkdir(exist_ok=True)

# Učitavanje svih 10 workera
worker_files = sorted(run_dir.glob("worker_*_runs.csv"))
assert len(worker_files) == 10, f"Pronađeno worker datoteka: {len(worker_files)}"

runs = pd.concat(
    [pd.read_csv(f, dtype={"architecture_id": str}) for f in worker_files],
    ignore_index=True
)

metrics = pd.read_csv(
    study / "graph_metrics_same_216.csv",
    dtype={"architecture_id": str}
)

split = pd.read_csv(
    study / "cifar100_full216_analysis_split.csv",
    dtype={"architecture_id": str}
)

# Sažetak pet treniranja po arhitekturi
agg_spec = {
    "completed_seeds": ("seed", "nunique"),
    "mean_test_acc": ("test_acc", "mean"),
    "std_test_acc": ("test_acc", "std"),
    "min_test_acc": ("test_acc", "min"),
    "max_test_acc": ("test_acc", "max"),
}

if "test_loss" in runs.columns:
    agg_spec["mean_test_loss"] = ("test_loss", "mean")
    agg_spec["std_test_loss"] = ("test_loss", "std")

if "best_epoch" in runs.columns:
    agg_spec["mean_best_epoch"] = ("best_epoch", "mean")

if "training_time_s" in runs.columns:
    agg_spec["mean_training_time_s"] = ("training_time_s", "mean")
    agg_spec["total_training_time_s"] = ("training_time_s", "sum")

summary = (
    runs.groupby("architecture_id", as_index=False)
        .agg(**agg_spec)
)

# Spajanje metrika, rezultata i unaprijed zaključane podjele
split_cols = ["architecture_id", "worker_id", "analysis_split", "used_in_pilot"]
split_cols = [c for c in split_cols if c in split.columns]

full = metrics.merge(
    summary,
    on="architecture_id",
    how="inner",
    validate="one_to_one"
).merge(
    split[split_cols],
    on="architecture_id",
    how="inner",
    validate="one_to_one"
)

runs_with_split = runs.merge(
    split[["architecture_id", "analysis_split", "used_in_pilot"]],
    on="architecture_id",
    how="inner",
    validate="many_to_one"
)

development = full[full["analysis_split"] == "development"].copy()
holdout = full[full["analysis_split"] == "final_holdout"].copy()

development_runs = runs_with_split[
    runs_with_split["analysis_split"] == "development"
].copy()

# Završne provjere
assert len(runs) == 1080
assert len(full) == 216
assert (full["completed_seeds"] == 5).all()
assert len(development) == 162
assert len(holdout) == 54
assert len(development_runs) == 810

# Potpuni rezultati ostaju u /storage i ne analiziraju se još
full.to_csv(archive / "full_summary_LOCKED.csv", index=False)
runs_with_split.to_csv(archive / "all_1080_runs_LOCKED.csv", index=False)
holdout.to_csv(archive / "final_holdout54_LOCKED.csv", index=False)
development.to_csv(archive / "development162.csv", index=False)

# Excel za sadašnju analizu u R-u: samo razvojni rezultati
excel_path = Path("/notebooks/cifar100_development162.xlsx")

readme = pd.DataFrame({
    "stavka": [
        "development_162",
        "development_810_runs",
        "graph_metrics_216",
        "split_manifest"
    ],
    "opis": [
        "162 arhitekture za razvoj i izbor prediktivnog modela",
        "Pojedinačni rezultati 162 × 5 seedova",
        "Topološke metrike svih arhitektura, bez rezultata točnosti",
        "Unaprijed zaključana podjela; final_holdout se ne analizira"
    ]
})

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    readme.to_excel(writer, sheet_name="README", index=False)
    development.to_excel(writer, sheet_name="development_162", index=False)
    development_runs.to_excel(
        writer, sheet_name="development_810_runs", index=False
    )
    metrics.to_excel(writer, sheet_name="graph_metrics_216", index=False)
    split[split_cols].to_excel(
        writer, sheet_name="split_manifest", index=False
    )

print("PROVJERA USPJEŠNA")
print("Ukupno:", len(full))
print("Development:", len(development))
print("Final holdout zaključan:", len(holdout))
print("Excel za R:", excel_path)

PROVJERA USPJEŠNA
Ukupno: 216
Development: 162
Final holdout zaključan: 54
Excel za R: /notebooks/cifar100_development162.xlsx


In [21]:
# Instanca 1
!python /storage/train_svhn_full216.py --worker-id 1 --workers 10 --pilot-size 216

Reused 1 completed SVHN run(s).
Worker 1: 22 architectures, 110 total seed runs.
SVHN data: train=45000, validation=5000, test=10000
Starting SVHN worker=1, architecture=A007, seed=22
I0000 00:00:1790679134.751048  285455 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
Finished A007, seed=22: test_acc=0.9048, best_epoch=31, time=2.0 min
Starting SVHN worker=1, architecture=A007, seed=33
Finished A007, seed=33: test_acc=0.9075, best_epoch=35, time=2.1 min
Starting SVHN worker=1, architecture=A007, seed=44
Finished A007, seed=44: test_acc=0.9040, best_epoch=34, time=2.2 min
Starting SVHN worker=1, architecture=A007, seed=55
Finished A007, seed=55: test_acc=0.9021, best_epoch=30, time=1.9 min
Starting SVHN worker=1, architecture=A011, seed=11
Finished A011, seed=11: test_acc=0.8931, best_epoch=60, time=4.4 min
Starting SVHN worker=1, architecture=A011, seed=22
Finished A011, seed=22: test_acc=0.8852, best_epoch=73, time

In [22]:
from pathlib import Path
import json
import pandas as pd

SEEDS = [11, 22, 33, 44, 55]
BROJ_WORKERA = 10

study_dir = Path("/storage/cnn_topology_study_same216")
output_dir = study_dir / "training_svhn_full216"
assignments_path = study_dir / "svhn_full216_assignments.csv"

# 1. Provjera datoteka
worker_files = sorted(output_dir.glob("worker_*_runs.csv"))
ocekivana_imena = [
    f"worker_{i:02d}_runs.csv"
    for i in range(1, BROJ_WORKERA + 1)
]

print("Broj worker datoteka:", len(worker_files))
print([p.name for p in worker_files])

assert [p.name for p in worker_files] == ocekivana_imena, (
    "Nedostaje worker datoteka ili postoji neočekivana datoteka."
)

# 2. Učitavanje assignments tablice
assignments = pd.read_csv(assignments_path)
assignments["architecture_id"] = assignments["architecture_id"].astype(str)
assignments["worker_id"] = assignments["worker_id"].astype(int)

assert len(assignments) == 216
assert assignments["architecture_id"].nunique() == 216
assert set(assignments["worker_id"]) == set(range(1, 11))

# 3. Učitavanje svih treninga
svi_runovi = []

for path in worker_files:
    df = pd.read_csv(path)
    df["architecture_id"] = df["architecture_id"].astype(str)
    df["worker_id"] = df["worker_id"].astype(int)
    df["seed"] = df["seed"].astype(int)
    df["izvorna_datoteka"] = path.name

    broj_duplikata = df.duplicated(
        ["architecture_id", "seed"]
    ).sum()

    print(
        f"{path.name}: redova={len(df)}, "
        f"arhitektura={df['architecture_id'].nunique()}, "
        f"duplikata={broj_duplikata}"
    )

    svi_runovi.append(df)

runs = pd.concat(svi_runovi, ignore_index=True)

# 4. Osnovne provjere
kriticne_kolone = [
    "dataset",
    "worker_id",
    "architecture_index",
    "architecture_id",
    "model_name",
    "seed",
    "best_epoch",
    "test_loss",
    "test_acc",
    "num_parameters_total",
]

nedostajuce_kolone = [
    x for x in kriticne_kolone
    if x not in runs.columns
]

assert not nedostajuce_kolone, (
    f"Nedostaju kolone: {nedostajuce_kolone}"
)

ukupno_duplikata = runs.duplicated(
    ["architecture_id", "seed"]
).sum()

print("\nUkupno redova:", len(runs))
print("Broj arhitektura:", runs["architecture_id"].nunique())
print("Broj seedova:", runs["seed"].nunique())
print("Ukupno duplikata:", ukupno_duplikata)
print(
    "Nedostajućih ključnih vrijednosti:",
    runs[kriticne_kolone].isna().sum().sum()
)

assert len(runs) == 1080
assert runs["architecture_id"].nunique() == 216
assert set(runs["seed"]) == set(SEEDS)
assert ukupno_duplikata == 0
assert runs[kriticne_kolone].notna().all().all()
assert runs["test_acc"].between(0, 1).all()
assert runs["test_loss"].ge(0).all()
assert runs["best_epoch"].ge(1).all()
assert runs["num_parameters_total"].gt(0).all()
assert runs["dataset"].eq("svhn").all()

# 5. Svaka arhitektura mora imati tačno pet seedova
po_arhitekturi = (
    runs.groupby("architecture_id")
    .agg(
        broj_redova=("seed", "size"),
        broj_seedova=("seed", "nunique"),
        mean_test_acc=("test_acc", "mean"),
        std_test_acc=("test_acc", "std"),
    )
)

assert po_arhitekturi["broj_redova"].eq(5).all()
assert po_arhitekturi["broj_seedova"].eq(5).all()

# Svaki seed mora biti korišten za svih 216 arhitektura
po_seedu = runs.groupby("seed")["architecture_id"].nunique()
assert po_seedu.eq(216).all()

# 6. Usporedba s assignments tablicom
seed_tablica = pd.DataFrame({"seed": SEEDS})
assignments_tmp = assignments[
    ["architecture_id", "worker_id"]
].copy()

assignments_tmp["_key"] = 1
seed_tablica["_key"] = 1

ocekivani_runovi = (
    assignments_tmp
    .merge(seed_tablica, on="_key")
    .drop(columns="_key")
)

stvarni_kljucevi = runs[
    ["architecture_id", "worker_id", "seed"]
].drop_duplicates()

usporedba = ocekivani_runovi.merge(
    stvarni_kljucevi,
    on=["architecture_id", "worker_id", "seed"],
    how="outer",
    indicator=True,
)

nedostajuci = usporedba[usporedba["_merge"] == "left_only"]
neocekivani = usporedba[usporedba["_merge"] == "right_only"]

print("Nedostajućih treninga:", len(nedostajuci))
print("Neočekivanih treninga:", len(neocekivani))

assert nedostajuci.empty
assert neocekivani.empty

# 7. Broj treninga po workeru
pregled_workera = pd.DataFrame({
    "ocekivano": (
        assignments.groupby("worker_id").size() * len(SEEDS)
    ),
    "dobiveno": runs.groupby("worker_id").size(),
})

print("\nTreninzi po workeru:")
print(pregled_workera)

assert pregled_workera["ocekivano"].eq(
    pregled_workera["dobiveno"]
).all()

# 8. Parametri moraju biti jednaki kroz seedove iste arhitekture
broj_verzija_parametara = (
    runs.groupby("architecture_id")["num_parameters_total"].nunique()
)

assert broj_verzija_parametara.eq(1).all()

# 9. Provjera metapodataka svih workera
for worker_id in range(1, BROJ_WORKERA + 1):
    metadata_path = (
        output_dir / f"worker_{worker_id:02d}_metadata.json"
    )

    assert metadata_path.exists(), (
        f"Nedostaje {metadata_path.name}"
    )

    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    assert metadata["dataset"] == "svhn"
    assert metadata["worker_id"] == worker_id
    assert set(metadata["seeds"]) == set(SEEDS)
    assert metadata["train_samples"] == 45000
    assert metadata["validation_samples"] == 5000
    assert metadata["test_samples"] == 10000

print(
    "\nPROVJERA USPJEŠNA: "
    "216 arhitektura × 5 seedova = 1080 SVHN treninga."
)

Broj worker datoteka: 10
['worker_01_runs.csv', 'worker_02_runs.csv', 'worker_03_runs.csv', 'worker_04_runs.csv', 'worker_05_runs.csv', 'worker_06_runs.csv', 'worker_07_runs.csv', 'worker_08_runs.csv', 'worker_09_runs.csv', 'worker_10_runs.csv']
worker_01_runs.csv: redova=110, arhitektura=22, duplikata=0
worker_02_runs.csv: redova=110, arhitektura=22, duplikata=0
worker_03_runs.csv: redova=110, arhitektura=22, duplikata=0
worker_04_runs.csv: redova=110, arhitektura=22, duplikata=0
worker_05_runs.csv: redova=110, arhitektura=22, duplikata=0
worker_06_runs.csv: redova=110, arhitektura=22, duplikata=0
worker_07_runs.csv: redova=105, arhitektura=21, duplikata=0
worker_08_runs.csv: redova=105, arhitektura=21, duplikata=0
worker_09_runs.csv: redova=105, arhitektura=21, duplikata=0
worker_10_runs.csv: redova=105, arhitektura=21, duplikata=0

Ukupno redova: 1080
Broj arhitektura: 216
Broj seedova: 5
Ukupno duplikata: 0
Nedostajućih ključnih vrijednosti: 0
Nedostajućih treninga: 0
Neočekivanih 

In [23]:
from pathlib import Path
import pandas as pd

study_dir = Path("/storage/cnn_topology_study_same216")
output_dir = study_dir / "training_svhn_full216"
notebooks_dir = Path("/notebooks")

# 1. Učitavanje 1080 treninga
worker_files = sorted(output_dir.glob("worker_*_runs.csv"))
runs = pd.concat(
    [pd.read_csv(path) for path in worker_files],
    ignore_index=True
)

runs["architecture_id"] = runs["architecture_id"].astype(str)
runs["seed"] = runs["seed"].astype(int)

# 2. Sažetak pet seedova za svaku arhitekturu
tocnost = (
    runs.groupby("architecture_id", as_index=False)
    .agg(
        mean_test_acc=("test_acc", "mean"),
        std_test_acc=("test_acc", "std"),
        min_test_acc=("test_acc", "min"),
        max_test_acc=("test_acc", "max"),
        mean_test_loss=("test_loss", "mean"),
        mean_best_val_acc=("best_val_acc", "mean"),
        mean_best_epoch=("best_epoch", "mean"),
        mean_training_time_s=("training_time_s", "mean"),
        num_parameters_total=("num_parameters_total", "first"),
        broj_seedova=("seed", "nunique"),
    )
)

assert len(tocnost) == 216
assert tocnost["broj_seedova"].eq(5).all()
assert tocnost[
    ["mean_test_acc", "std_test_acc"]
].notna().all().all()

# 3. Učitavanje zajedničkih graf-metrika
metrics_path = (
    study_dir / "graph_metrics_same_216_modularity5.csv"
)

metrike = pd.read_csv(metrics_path)
metrike["architecture_id"] = metrike["architecture_id"].astype(str)

assert len(metrike) == 216
assert metrike["architecture_id"].nunique() == 216

potrebne_metrike = [
    "architecture_id",
    "avg_degree",
    "degree_entropy",
    "normalized_degree_entropy",
    "assortativity_coefficient",
    "modularity_best5",
]

nedostajuce = [
    x for x in potrebne_metrike
    if x not in metrike.columns
]

assert not nedostajuce, (
    f"Nedostaju graf-metrike: {nedostajuce}"
)

# Podjelu ćemo dodati iz zaključane CIFAR-10 tablice
metrike = metrike.drop(
    columns=["analysis_split"],
    errors="ignore"
)

# 4. Učitavanje iste zaključane podjele 162/54
cifar10_reference_path = (
    notebooks_dir / "cifar10_full216_complete.xlsx"
)

cifar10_reference = pd.read_excel(cifar10_reference_path)
cifar10_reference["architecture_id"] = (
    cifar10_reference["architecture_id"].astype(str)
)

assert "analysis_split" in cifar10_reference.columns

podjela = (
    cifar10_reference[
        ["architecture_id", "analysis_split"]
    ]
    .drop_duplicates()
)

assert len(podjela) == 216
assert podjela["architecture_id"].nunique() == 216

# 5. Spajanje metrika, SVHN rezultata i podjele
svhn_full = (
    metrike
    .merge(
        tocnost,
        on="architecture_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        podjela,
        on="architecture_id",
        how="inner",
        validate="one_to_one",
    )
)

if "architecture_index" in svhn_full.columns:
    svhn_full = svhn_full.sort_values(
        "architecture_index"
    ).reset_index(drop=True)
else:
    svhn_full = svhn_full.sort_values(
        "architecture_id"
    ).reset_index(drop=True)

development = svhn_full.loc[
    svhn_full["analysis_split"].eq("development")
].copy()

final_holdout = svhn_full.loc[
    svhn_full["analysis_split"].eq("final_holdout")
].copy()

# 6. Završne provjere
assert len(svhn_full) == 216
assert len(development) == 162
assert len(final_holdout) == 54

assert set(development["architecture_id"]).isdisjoint(
    set(final_holdout["architecture_id"])
)

assert svhn_full["mean_test_acc"].notna().all()
assert svhn_full["mean_test_acc"].between(0, 1).all()

print("Podjela:")
print(svhn_full["analysis_split"].value_counts())

print(
    "\nRaspon prosječne SVHN testne točnosti:",
    svhn_full["mean_test_acc"].min(),
    "-",
    svhn_full["mean_test_acc"].max(),
)

print(
    "Prosječna točnost svih arhitektura:",
    svhn_full["mean_test_acc"].mean(),
)

# 7. Spremanje Excel datoteka
full_path = notebooks_dir / "svhn_full216_complete.xlsx"
development_path = notebooks_dir / "svhn_development162.xlsx"
holdout_path = notebooks_dir / "svhn_final_holdout54.xlsx"

svhn_full.to_excel(
    full_path,
    sheet_name="full_216",
    index=False,
)

development.to_excel(
    development_path,
    sheet_name="development_162",
    index=False,
)

final_holdout.to_excel(
    holdout_path,
    sheet_name="final_holdout_54",
    index=False,
)

print("\nSpremljeno:")
print(full_path)
print(development_path)
print(holdout_path)

print(
    "\nSPAJANJE USPJEŠNO: "
    "216 ukupno, 162 development i 54 final holdout."
)

Podjela:
analysis_split
development      162
final_holdout     54
Name: count, dtype: int64

Raspon prosječne SVHN testne točnosti: 0.8006200075149537 - 0.9286000013351441
Prosječna točnost svih arhitektura: 0.896755648266386

Spremljeno:
/notebooks/svhn_full216_complete.xlsx
/notebooks/svhn_development162.xlsx
/notebooks/svhn_final_holdout54.xlsx

SPAJANJE USPJEŠNO: 216 ukupno, 162 development i 54 final holdout.
